# Preprocessing – Smoking & Drinking Dataset

## Objective
Build a reproducible preprocessing pipeline to transform raw health data into a
clean, model-ready dataset for alcohol consumption prediction.

## Input
Raw dataset:
- `data/raw/smoking_driking_dataset_Ver01.csv`

## Output
Processed dataset and preprocessing artifacts to be used in downstream modeling.

## Data loading and initial setup

### Imports & configuración

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

RANDOM_SEED = 42

### Define data paths

In [3]:
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


### Load raw dataset

In [4]:
raw_path = RAW_DIR / "smoking_driking_dataset_Ver01.csv"

if not raw_path.exists():
    raise FileNotFoundError(
        f"Raw dataset not found at: {raw_path}\n"
        "Please download it from Kaggle and place it in data/raw/"
    )

df_raw = pd.read_csv(raw_path)
df = df_raw.copy()

df.shape

(991346, 24)

### Sanity check

In [5]:
df.head()

,sex,age,height,weight,waistline,sight_left,sight_right,hear_left,hear_right,SBP,...,LDL_chole,triglyceride,hemoglobin,urine_protein,serum_creatinine,SGOT_AST,SGOT_ALT,gamma_GTP,SMK_stat_type_cd,DRK_YN
0,Male,35,170,75,90.0,1.0,1.0,1.0,1.0,120.0,...,126.0,92.0,17.1,1.0,1.0,21.0,35.0,40.0,1.0,Y
1,Male,30,180,80,89.0,0.9,1.2,1.0,1.0,130.0,...,148.0,121.0,15.8,1.0,0.9,20.0,36.0,27.0,3.0,N
2,Male,40,165,75,91.0,1.2,1.5,1.0,1.0,120.0,...,74.0,104.0,15.8,1.0,0.9,47.0,32.0,68.0,1.0,N
3,Male,50,175,80,91.0,1.5,1.2,1.0,1.0,145.0,...,104.0,106.0,17.6,1.0,1.1,29.0,34.0,18.0,1.0,N
4,Male,50,165,60,80.0,1.0,1.2,1.0,1.0,138.0,...,117.0,104.0,13.8,1.0,0.8,19.0,12.0,25.0,1.0,N


## Target and feature definition

In this step, the target variable is defined and features are separated from labels.
No preprocessing is applied to features at this stage; encoding, imputation, and
scaling will be handled later using reproducible pipelines.

### Define target variable

In [6]:
TARGET_COL = "DRK_YN"

# Validate target column presence
if TARGET_COL not in df.columns:
    raise KeyError(f"Target column '{TARGET_COL}' not found in the dataset.")


### Separate features and labels

In [7]:
# Separate features and target (raw features, no preprocessing applied)
X = df.drop(columns=[TARGET_COL]).copy()
y_raw = df[TARGET_COL].copy()

X.shape, y_raw.shape

((991346, 23), (991346,))

### Map target to binary

In [8]:
# Map target variable from Y/N to binary labels

y = y_raw.map({"N": 0, "Y": 1})

# Sanity checks
if y.isna().any():
    invalid = y_raw[ y.isna() ].value_counts()
    raise ValueError(f"Unexpected values in target '{TARGET_COL}':\n{invalid}")

y.value_counts(), y.value_counts(normalize=True) * 100


(DRK_YN
 0    495858
 1    495488
 Name: count, dtype: int64,
 DRK_YN
 0    50.018661
 1    49.981339
 Name: proportion, dtype: float64)

### Identify feature types

In [9]:
# Identify categorical and numerical feature columns
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

categorical_cols, len(categorical_cols), len(numeric_cols)


(['sex'], 1, 22)

### Feature cardinality and encoding rationale

In [12]:
# Inspect cardinality of categorical and ordinal-like features
X[categorical_cols + ["SMK_stat_type_cd"]].nunique()

sex                 2
SMK_stat_type_cd    3
dtype: int64

### Feature cardinality and encoding rationale

- `sex` has low cardinality and represents a nominal category, making it suitable
  for one-hot encoding.
- `SMK_stat_type_cd` has three discrete values with an inherent order
  (never, former, current smoker), supporting its treatment as an ordinal numerical
  feature rather than a nominal categorical variable.

## Invalid values and placeholder handling

Based on the EDA findings, this step identifies physiologically implausible values and
known placeholder codes (e.g., 999, 9999) and converts them to missing values (`NaN`).
No imputation is applied at this stage.


### Snapshot of suspicious placeholder codes before replacement

In [16]:
# Snapshot of suspicious placeholder codes before replacement
placeholder_checks = {
    "waistline_999": (X["waistline"] == 999).sum(),
    "gamma_GTP_999": (X["gamma_GTP"] == 999).sum(),
    "SGOT_AST_9999": (X["SGOT_AST"] == 9999).sum(),
}

pd.Series(placeholder_checks, name="count")


waistline_999     57
gamma_GTP_999    239
SGOT_AST_9999      1
Name: count, dtype: int64

### Observations

The snapshot reveals a non-negligible presence of explicit placeholder codes in the
dataset. In particular:

- The `waistline` variable contains a small number of records coded as `999`,
  which is physiologically implausible.
- The `gamma_GTP` variable shows a higher frequency of the placeholder value `999`,
  suggesting systematic use of this code to represent invalid or missing measurements.
- Only a single occurrence of the extreme placeholder value `9999` is observed in
  `SGOT_AST`.

These observations support the decision to explicitly treat these values as invalid
and convert them to missing values prior to imputation.

### Define placeholder and physiological rules

In [18]:
# Rules: exact placeholder values to be treated as missing
PLACEHOLDER_VALUES = {
    "waistline": [999],
    "gamma_GTP": [999],
    "SGOT_AST": [9999],
}

# Rules: physiologically implausible lower bounds (domain-informed)
# Note: thresholds are intentionally conservative; they can be refined later.
LOWER_BOUNDS = {
    "waistline": 30,  # cm (values like 8 are not plausible)
}

PLACEHOLDER_VALUES, LOWER_BOUNDS


({'waistline': [999], 'gamma_GTP': [999], 'SGOT_AST': [9999]},
 {'waistline': 30})

The rules defined above are intentionally conservative and target only values that are clearly incompatible with physiological measurements or known placeholder codes.
Extreme but plausible clinical values are retained to preserve rare but informative observations.

### Utility functions for invalid value handling

In [19]:
from typing import List

def replace_with_nan_exact(
    df: pd.DataFrame,
    col: str,
    values: List[float]
) -> int:
    """Replace exact placeholder values with NaN and return number of replacements."""
    before = df[col].isna().sum()
    df.loc[df[col].isin(values), col] = np.nan
    after = df[col].isna().sum()
    return after - before

def apply_lower_bound_nan(
    df: pd.DataFrame,
    col: str,
    lower: float
) -> int:
    """Replace values strictly below a lower bound with NaN."""
    before = df[col].isna().sum()
    df.loc[df[col] < lower, col] = np.nan
    after = df[col].isna().sum()
    return after - before



### Apply placeholder and physiological rules

In [20]:
# Apply placeholder and physiological rules and log replacements
replacements = []

for col, vals in PLACEHOLDER_VALUES.items():
    if col in X.columns:
        n = replace_with_nan_exact(X, col, vals)
        replacements.append((col, "exact_placeholder", n))

for col, lb in LOWER_BOUNDS.items():
    if col in X.columns:
        n = apply_lower_bound_nan(X, col, lb)
        replacements.append((col, "lower_bound", n))

replacements_df = (
    pd.DataFrame(replacements, columns=["column", "rule", "n_replaced"])
    .sort_values(by="n_replaced", ascending=False)
    .reset_index(drop=True)
)
replacements_df

,column,rule,n_replaced
0,gamma_GTP,exact_placeholder,239
1,waistline,exact_placeholder,57
2,waistline,lower_bound,2
3,SGOT_AST,exact_placeholder,1


### Missing values after placeholder handling

In [21]:
missing_after = (X.isna().mean() * 100).round(3).sort_values(ascending=False)
missing_after.head(10)

gamma_GTP      0.024
waistline      0.006
age            0.000
height         0.000
weight         0.000
sight_left     0.000
sex            0.000
sight_right    0.000
hear_left      0.000
SBP            0.000
dtype: float64

At this stage, invalid placeholder values and physiologically implausible observations
have been converted to missing values (`NaN`). The introduction of missing values is
limited to a small subset of variables affected by explicit placeholder codes or clearly
invalid measurements, confirming that the applied cleaning rules are conservative and
do not substantially reduce the available information in the dataset.

These missing values will be handled in the next step using an explicit imputation
strategy implemented within a reproducible preprocessing pipeline.

### Additional considerations on extreme values

Several biochemical variables (e.g., cholesterol fractions, triglycerides, and liver
enzymes) exhibit extremely high maximum values that are far removed from the central
distribution. These observations are not treated as invalid placeholders, as they may
represent rare but clinically plausible conditions rather than data entry errors.

Accordingly, these values are retained at this stage and will be handled using robust
preprocessing techniques (e.g., robust scaling or transformations) rather than hard
exclusion.

## Missing values strategy

In this step, missing values introduced during placeholder handling are addressed using
explicit imputation rules. Imputation is performed as part of a preprocessing pipeline
to ensure reproducibility and prevent data leakage.


In [24]:
# Percentage of missing values per feature
missing_pct = (X.isna().mean() * 100).round(3)
missing_pct[missing_pct > 0].sort_values(ascending=False)


gamma_GTP    0.024
waistline    0.006
dtype: float64

### Imputation rationale

- Numerical features are imputed using the median to reduce sensitivity to skewed distributions and extreme values.
- Categorical features are imputed using the most frequent value.
- Imputation is performed within a preprocessing pipeline to avoid information leakage between training and validation data.


In [25]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

num_imputer, cat_imputer

(SimpleImputer(strategy='median'), SimpleImputer(strategy='most_frequent'))

In [26]:
len(numeric_cols), len(categorical_cols), numeric_cols[:5], categorical_cols

(22, 1, ['age', 'height', 'weight', 'waistline', 'sight_left'], ['sex'])

At this stage, the imputation strategy has been defined but not yet applied. Missing
value handling will be executed within a full preprocessing pipeline to ensure proper
fit/transform separation.

## Encoding and scaling

This step defines how categorical and numerical features are encoded and scaled prior
to modeling. All transformations are designed to be applied within a preprocessing
pipeline to ensure reproducibility and avoid data leakage.

### Encoding and scaling rationale

- Categorical variables with nominal meaning are encoded using one-hot encoding.
- Ordinal-coded variables are retained as numerical features.
- Numerical features are scaled using robust statistics to reduce sensitivity to
  skewed distributions and extreme values.

In [27]:
from sklearn.preprocessing import OneHotEncoder

cat_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

cat_encoder


,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",None
,"max_cate

In [28]:
from sklearn.preprocessing import RobustScaler

num_scaler = RobustScaler()

num_scaler


,"with_centering with_centering: bool, default=TrueIf `True`, center the data before scaling.This will cause :meth:`transform` to raise an exception when attemptedon sparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_scaling with_scaling: bool, default=TrueIf `True`, scale the data to interquartile range.",True
,"quantile_range quantile_range: tuple (q_min, q_max), 0.0 < q_min < q_max < 100.0, default=(25.0, 75.0)Quantile range used to calculate `scale_`. By default this is equal tothe IQR, i.e., `q_min` is the first quantile and `q_max` is the thirdquantile... versionadded:: 0.18","(25.0, ...)"
,"copy copy: bool, default=TrueIf `False`, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"unit_variance unit_variance: bool, default=FalseIf `True`, scale data so that normally distributed features have avariance of 1. In general, if the difference between the x-values of`q_max` and `q_min` for a standard normal distribution is greaterthan 1, the dataset will be scaled down. If less than 1, the datasetwill be scaled up... versionadded:: 0.24",False


In [29]:
categorical_cols, numeric_cols[:10]

(['sex'],
 ['age',
  'height',
  'weight',
  'waistline',
  'sight_left',
  'sight_right',
  'hear_left',
  'hear_right',
  'SBP',
  'DBP'])

At this stage, encoding and scaling strategies have been defined but not yet applied.
These transformations will be combined with imputation and executed within a unified
preprocessing pipeline in the next step.

## Train/validation split and preprocessing pipeline

In this step, the dataset is split into training and validation sets using a stratified
strategy. All preprocessing steps (imputation, encoding, scaling) are combined into a
single pipeline to ensure reproducibility and prevent data leakage.


### Why stratified splitting?

A stratified train/validation split is used to preserve the original class distribution
of the target variable (`DRK_YN`) in both subsets. This is particularly important for
classification tasks, as it ensures that performance metrics computed on the validation
set are representative and not biased by class proportion shifts.

In [30]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_SEED
)

X_train.shape, X_val.shape, y_train.value_counts(normalize=True), y_val.value_counts(normalize=True)


((793076, 23),
 (198270, 23),
 DRK_YN
 0    0.500187
 1    0.499813
 Name: proportion, dtype: float64,
 DRK_YN
 0    0.500187
 1    0.499813
 Name: proportion, dtype: float64)

### Build preprocessing pipeline (ColumnTransformer)

We define separate preprocessing steps for numerical and categorical features and
combine them using a `ColumnTransformer`.

In [31]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_transformer = Pipeline(
    steps=[
        ("imputer", num_imputer),
        ("scaler", num_scaler),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", cat_imputer),
        ("encoder", cat_encoder),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

preprocessor


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

### Fit preprocessing pipeline on training data

The preprocessing pipeline is fitted exclusively on the training set and then applied
to both training and validation data.

In [32]:
# Fit only on training data to prevent leakage
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)

X_train_proc.shape, X_val_proc.shape

((793076, 24), (198270, 24))

The resulting matrices contain fully preprocessed features and are ready for use in downstream modeling tasks.

### Retrieve feature names after preprocessing

After applying encoding and scaling, feature names are extracted to support interpretability and model inspection.

In [33]:
feature_names = preprocessor.get_feature_names_out()
len(feature_names), feature_names[:10]


(24,
 array(['age', 'height', 'weight', 'waistline', 'sight_left',
        'sight_right', 'hear_left', 'hear_right', 'SBP', 'DBP'],
       dtype=object))

The final feature space reflects the expansion introduced by categorical encoding and the transformations applied to numerical variables.

### Persist preprocessing artifacts

Preprocessing artifacts and transformed datasets are saved to disk for reuse in downstream modeling steps.

In [34]:
from pathlib import Path
import joblib

ARTIFACTS_DIR = Path("../artifacts")
PROCESSED_DIR = Path("../data/processed")

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(preprocessor, ARTIFACTS_DIR / "preprocessor.joblib")

np.save(PROCESSED_DIR / "X_train.npy", X_train_proc)
np.save(PROCESSED_DIR / "X_val.npy", X_val_proc)
np.save(PROCESSED_DIR / "y_train.npy", y_train.values)
np.save(PROCESSED_DIR / "y_val.npy", y_val.values)

joblib.dump(feature_names, ARTIFACTS_DIR / "feature_names.joblib")




['..\\artifacts\\feature_names.joblib']

In [35]:
np.isnan(X_train_proc).sum(), np.isnan(X_val_proc).sum()


(np.int64(0), np.int64(0))

At this stage, the dataset has been split into training and validation sets, and all preprocessing steps have been encapsulated in a unified pipeline and persisted as reusable artifacts. The resulting preprocessed datasets and associated metadata are ready to be consumed by downstream modeling workflows.
